In [9]:
import pandas as pd

datos = pd.read_excel("[HackMTY2025]_ConsumptionPrediction_Dataset_v1.xlsx")

In [4]:
datos.head()

,Flight_ID,Origin,Date,Flight_Type,Service_Type,Passenger_Count,Product_ID,Product_Name,Standard_Specification_Qty,Quantity_Returned,Quantity_Consumed,Unit_Cost,Crew_Feedback
0,AM109,DOH,2025-09-26,medium-haul,Retail,272,BRD001,Bread Roll Pack,62,7,55,0.35,NaN
1,AM109,DOH,2025-09-26,medium-haul,Retail,272,CRK075,Butter Cookies 75g,74,14,60,0.75,NaN
2,AM109,DOH,2025-09-26,medium-haul,Retail,272,DRK023,Sparkling Water 330ml,125,30,95,0.45,NaN
3,AM109,DOH,2025-09-26,medium-haul,Retail,272,DRK024,Still Water 500ml,110,19,91,0.50,NaN
4,LX110,DOH,2025-09-26,medium-haul,Pick & Pack,272,BRD001,Bread Roll Pack,177,58,119,0.35,NaN


In [5]:
print(datos.isnull().sum())

Flight_ID                       0
Origin                          0
Date                            0
Flight_Type                     0
Service_Type                    0
Passenger_Count                 0
Product_ID                      0
Product_Name                    0
Standard_Specification_Qty      0
Quantity_Returned               0
Quantity_Consumed               0
Unit_Cost                       0
Crew_Feedback                 723
dtype: int64


In [10]:
datos.drop('Crew_Feedback', axis=1, inplace=True)
datos.drop('Flight_ID', axis=1, inplace=True)
datos.drop('Service_Type', axis=1, inplace=True)
datos.drop('Product_ID', axis=1, inplace=True)
datos.drop('Unit_Cost', axis=1, inplace=True)

In [11]:
print(datos.isnull().sum())

Origin                        0
Date                          0
Flight_Type                   0
Passenger_Count               0
Product_Name                  0
Standard_Specification_Qty    0
Quantity_Returned             0
Quantity_Consumed             0
dtype: int64


In [14]:
print(datos.columns)

Index(['Origin', 'Date', 'Flight_Type', 'Passenger_Count', 'Product_Name',
       'Standard_Specification_Qty', 'Quantity_Returned', 'Quantity_Consumed'],
      dtype='object')


In [15]:
# Convertir la columna 'Date' a formato de fecha
datos['Date'] = pd.to_datetime(datos['Date'])

# Extraer características relevantes
datos['Year'] = datos['Date'].dt.year
datos['Month'] = datos['Date'].dt.month
datos['DayOfWeek'] = datos['Date'].dt.dayofweek # Lunes=0, Domingo=6
datos = datos.drop('Date', axis=1) # Ya no necesitamos la columna original

In [17]:
# 'Product_Name' es uno de nuestros objetivos (target), lo manejaremos después.
# Primero codificamos las características de entrada (features).
datos = pd.get_dummies(datos, columns=['Origin', 'Flight_Type'])

In [18]:
# Convertimos el nombre del producto a un código numérico (esto será nuestra salida categórica)
datos['Product_Code'] = datos['Product_Name'].astype('category').cat.codes
product_mapping = dict(enumerate(datos['Product_Name'].astype('category').cat.categories)) # Guardamos el mapeo para después

# Separar las salidas
y_producto = datos['Product_Code']
y_cantidad = datos['Quantity_Consumed']

# Separar las entradas y eliminar las columnas que son 'targets' o ya no sirven
X = datos.drop(['Product_Name', 'Product_Code', 'Quantity_Consumed', 'Quantity_Returned'], axis=1) # Asumimos que Returned no sirve para predecir el consumo

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_producto_train, y_producto_test, y_cantidad_train, y_cantidad_test = train_test_split(
    X, y_producto, y_cantidad, test_size=0.2, random_state=42
)

In [22]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout

# Definimos las entradas del modelo
input_layer = Input(shape=(X_train.shape[1],), name='input_features')

# Capas ocultas (el "cerebro" del modelo)
shared_layers = Dense(128, activation='relu')(input_layer)
shared_layers = Dropout(0.2)(shared_layers) # Dropout ayuda a prevenir sobreajuste
shared_layers = Dense(64, activation='relu')(shared_layers)

# ---- Cabeza de Salida 1: Predicción del Producto (Clasificación) ----
num_products = len(product_mapping) # Número de productos únicos
product_output = Dense(num_products, activation='softmax', name='product_output')(shared_layers)

# ---- Cabeza de Salida 2: Predicción de la Cantidad (Regresión) ----
quantity_output = Dense(1, activation='linear', name='quantity_output')(shared_layers)

# Unimos todo en un solo modelo
model = Model(inputs=input_layer, outputs=[product_output, quantity_output])

# Vemos un resumen de la arquitectura
model.summary()

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
# Compilamos el modelo con dos funciones de pérdida
model.compile(optimizer='adam',
              loss={
                  'product_output': 'sparse_categorical_crossentropy',
                  'quantity_output': 'mean_squared_error'
              },
              metrics={
                  'product_output': 'accuracy' # Nos interesa la precisión para el producto
              })

# Entrenamos el modelo
history = model.fit(
    X_train,
    {'product_output': y_producto_train, 'quantity_output': y_cantidad_train},
    epochs=50,  # Número de veces que el modelo verá todos los datos
    batch_size=32,
    validation_split=0.2 # Usamos parte de los datos de entrenamiento para validar
)

In [ ]:
# Evaluar el modelo
loss, product_loss, quantity_loss, product_accuracy = model.evaluate(
    X_test, [y_producto_test, y_cantidad_test]
)
print(f"Precisión del producto: {product_accuracy * 100:.2f}%")
print(f"Error en la cantidad (MSE): {quantity_loss}")

# Para hacer una predicción con datos nuevos
# (debes preprocesar los datos nuevos de la misma forma que los de entrenamiento)
predicciones = model.predict(X_test)
predicted_products = np.argmax(predicciones[0], axis=1)
predicted_quantities = predicciones[1]

# Ver las primeras 5 predicciones
for i in range(5):
    producto_real = product_mapping[y_producto_test.iloc[i]]
    producto_predicho = product_mapping[predicted_products[i]]
    print(f"Predicción {i+1}:")
    print(f"  -> Producto: Real='{producto_real}', Predicho='{producto_predicho}'")
    print(f"  -> Cantidad: Real={y_cantidad_test.iloc[i]}, Predicha={predicted_quantities[i][0]:.2f}\n")